# Advanced harness engineering: a fair MetaHarness evaluation

This notebook adapts the experimental-design lessons in MemoRizz
`examples/metaharness/06_fair_harness_comparison_results.ipynb` into a
self-contained course lab. It imports only the latest installed MemoRizz
PyPI wheel. The safe default performs a no-network mechanics audit; an
explicitly enabled and separately approved cell can run one GPT-5.5 call
and one Claude Opus 5 call under matched envelopes.

## Industry use case — fintech authorization-policy repair

A regulated fintech platform is choosing how to review code changes that
affect authorization expiry, tenant isolation, approval replay, and sandbox
egress. The deployment decision is not “which model wrote nicer prose?” It
is which *execution strategy* delivers the required task checks at an
acceptable cost and latency while preserving memory scope, approvals,
isolation, host verification, and auditability.

## Harness and evaluator component map

| Component | Role in the experiment | Failure it prevents |
|---|---|---|
| Frozen protocol | Hash tasks, arms, models, metrics, and order before spend | Outcome-driven redesign |
| Task suite | Four independent policy-repair families with hidden checks | Treating reruns as independent samples |
| Execution arms | Direct, MemoRizz-wrapped, always-panel, risk-routed | Confounding wrapper and coordination value |
| Provider stratum | Filesystem and Oracle for the same strategy | Calling storage differences orchestration value |
| Host verifier | Executable, task-native checks | Rewarding plausible but incorrect prose |
| Accounting | Calls, tokens, latency, cost, failures | Hiding retries or failed judge spend |
| Diagnostic judge | Identity-blind secondary signal only | Letting one subjective scalar override tests |
| Claim policy | `winner_allowed=false` for smoke runs | Generalizing beyond the observed task distribution |

```mermaid
flowchart TB
  Suite[Four independent fintech tasks] --> P{Memory provider}
  P --> FS[Filesystem]
  P --> ORA[Oracle AI Database]
  FS & ORA --> S{Six execution strategies}
  S --> D1[Direct GPT-5.5]
  S --> W1[MemoRizz to GPT-5.5]
  S --> D2[Direct Opus 5]
  S --> W2[MemoRizz to Opus 5]
  S --> Full[Always-on panel]
  S --> Routed[Preregistered risk route]
  D1 & W1 & D2 & W2 & Full & Routed --> Gates[Fail-closed host checks]
  Gates --> Effects[Paired effects by task]
  Effects --> Boundary[Bounded interpretation]
```


## Part I — establish provenance and freeze the protocol

### 1. Verify the published runtime

Reproducibility starts with the distributions actually imported. No cell
adds a MemoRizz checkout or `src` directory to `sys.path`.


In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path
from pprint import pprint

COURSE_ROOT = Path.cwd().resolve()
while COURSE_ROOT != COURSE_ROOT.parent and not (COURSE_ROOT / "part_1" / "advanced").exists():
    COURSE_ROOT = COURSE_ROOT.parent
if not (COURSE_ROOT / "part_1" / "advanced").exists():
    raise RuntimeError("Start Jupyter inside the agent_harness_course repository")
sys.path.insert(0, str(COURSE_ROOT))

print({
    "python": sys.version.split()[0],
    "langgraph": version("langgraph"),
    "langgraph-oracledb": version("langgraph-oracledb"),
    "oracleagentmemory": version("oracleagentmemory"),
    "oracledb": version("oracledb"),
    "memorizz": version("memorizz"),
    "tavily-python": version("tavily-python"),
    "anthropic": version("anthropic"),
    "e2b": version("e2b"),
    "e2b-code-interpreter": version("e2b-code-interpreter"),
})


In [ ]:
import memorizz

assert version("memorizz") == "0.6.3"
assert "site-packages" in str(Path(memorizz.__file__).resolve())
print("MemoRizz provenance:", Path(memorizz.__file__).resolve())


### 2. Build the preregistered 2 × 6 manifest

The manifest contains 12 arms: six strategies under Filesystem and Oracle.
Two reverse-order repeats give every arm the same mean execution position.
The protocol fingerprint excludes no decision-bearing field; changing the
suite, models, metrics, or order creates a different protocol.


In [ ]:
from part_1.advanced.shared.evaluation import (
    mechanics_artifact,
    protocol_manifest,
    source_notebook_lessons,
)

manifest = protocol_manifest(repeats=2, seed=20260825)
pprint({
    "protocol": manifest["protocol"],
    "fingerprint": manifest["protocol_fingerprint"],
    "arms": len(manifest["arms"]),
    "tasks": len(manifest["task_suite"]),
    "models": manifest["models"],
    "winner_allowed": manifest["winner_allowed"],
})
assert len(manifest["arms"]) == 12
assert manifest["winner_allowed"] is False
assert manifest["paper_comparable"] is False


### 3. Inspect balance and task independence

Reverse order controls the linear position effect, not every time effect.
Vendor drift, cache warming, transient load, and nonlinear time trends can
remain. More importantly, the sample unit is an independent task. Running
one task twice improves measurement precision but does not create two
independent examples.

```mermaid
flowchart LR
  Protocol[Protocol hash] --> Order1[Forward block]
  Protocol --> Order2[Reverse block]
  Order1 & Order2 --> Position[Equal mean arm position]
  Position --> Pair[Pair within the same task]
  Pair --> TaskEffect[One task-level effect]
  TaskEffect --> SuiteEffect[Aggregate across independent tasks]
  Rerun[Repeated same task] -. not a new sample .-> TaskEffect
```


In [ ]:
positions = {arm: [] for arm in manifest["arms"]}
for order in manifest["execution_orders"]:
    for position, arm in enumerate(order, start=1):
        positions[arm].append(position)
mean_positions = {
    arm: sum(values) / len(values) for arm, values in positions.items()
}
pprint({"unique_mean_positions": sorted(set(mean_positions.values()))})
assert set(mean_positions.values()) == {6.5}
pprint([
    {
        "task_id": task["task_id"],
        "family": task["family"],
        "risk": task["risk"],
        "hidden_checks": len(task["hidden_checks"]),
    }
    for task in manifest["task_suite"]
])


## Part II — define causal questions before comparing rows

### 4. Separate the estimands

An estimand is the exact contrast we intend to interpret. Direct GPT-5.5
versus MemoRizz→GPT-5.5 estimates wrapper overhead while holding the model
fixed. An always-on panel versus a wrapped single model estimates the
incremental value of a second reviewer. A risk-routed panel versus the
always-on panel estimates a routing policy, and Oracle versus Filesystem is
meaningful only for the same execution strategy.

```mermaid
flowchart TB
  Model[Model choice] --> Outcome[Quality / latency / cost]
  Wrapper[MemoRizz wrapper] --> Outcome
  Coordination[Second reviewer] --> Outcome
  Routing[Risk rule] --> Outcome
  Provider[Memory provider] --> Outcome
  Task[Task difficulty] --> Outcome
  Time[Execution order / vendor state] --> Outcome
  Compare[Valid paired contrast] --> Outcome
  Compare -. holds other causes fixed .-> Model
  Compare -. holds other causes fixed .-> Provider
```


In [ ]:
pprint(manifest["estimands"])
for arm, spec in sorted(manifest["arms"].items()):
    print(
        f"{arm:<34} {spec['interface']:<8} "
        f"{','.join(spec['models']):<28} {spec['call_policy']}"
    )


### 5. Use task-native checks as the primary metric

Policy repairs have observable requirements. Each clean workspace receives
the same source fixture and hidden verification command. A candidate is
fully correct only when all checks for that task pass. Micro pass rate and
fully-correct tasks remain interpretable even when an LLM judge fails.

An identity-blind judge may diagnose clarity or evidence quality, but the
evaluator must invalidate its scalar if it returns the wrong scale,
considers verbosity not present in the rubric, leaks model identity, or
fails to account for a provider error. “The judge returned JSON” is not a
validity argument.


## Part III — execute a no-cost MemoRizz mechanics audit

### 6. Initialize the real PyPI interfaces

The deterministic audit creates a disposable MemoRizz Filesystem provider,
durable SQLite run and approval stores, a Git workspace, and real
`MetaHarness.compare()` and `run_plan()` calls. Its adapters are fixed so
it tests the control plane without model or network variance.


In [ ]:
import tempfile
from getpass import getpass

from part_1.advanced.shared.metaharness_demo import MetaHarnessCourse

EVALUATION_LIVE = os.environ.get("ADVANCED_EVALUATION_LIVE", "0") == "1"
if EVALUATION_LIVE:
    evaluation_openai_key = getpass("OpenAI key for the approved GPT-5.5 smoke arm (hidden): ").strip()
    evaluation_anthropic_key = getpass("Anthropic key for the approved Opus 5 smoke arm (hidden): ").strip()
    if not evaluation_openai_key or not evaluation_anthropic_key:
        raise RuntimeError("Both hidden keys are required for the paid smoke arms")
else:
    evaluation_openai_key = ""
    evaluation_anthropic_key = ""

EVAL_ROOT = Path(tempfile.mkdtemp(prefix="advanced-fair-evaluation-"))
evaluation_course = MetaHarnessCourse(
    EVAL_ROOT,
    openai_api_key=evaluation_openai_key,
    anthropic_api_key=evaluation_anthropic_key,
    e2b_api_key="",
)
del evaluation_openai_key, evaluation_anthropic_key
print("Execution profile:", "paid paired smoke" if EVALUATION_LIVE else "deterministic no-network mechanics")


### 7. Run and validate the mechanics artifact

The three rows are a grounded single harness, an independent two-harness
comparison, and a serial adversarial plan. They report real MemoRizz
routing, context fingerprints, verification, normalized usage, calls, and
measured latency. They deliberately do not claim to measure either
frontier model.

```mermaid
flowchart LR
  Task[Same task + workspace] --> Single[Grounded single]
  Task --> Pair[Independent pair]
  Task --> Plan[Serial adversarial plan]
  Single & Pair & Plan --> Ctx[Context parity]
  Single & Pair & Plan --> Host[Host verification]
  Single & Pair & Plan --> Usage[Calls / tokens / latency]
  Ctx & Host & Usage --> Valid{Validity gates}
  Valid -->|pass| Artifact[Mechanics artifact]
  Valid -->|fail| Stop[No comparison]
```


In [ ]:
mechanics = mechanics_artifact(evaluation_course)
pprint(mechanics["validity_gates"])
pprint(mechanics["rows"])
pprint(mechanics["paired_deltas"])
assert mechanics["valid"] is True
assert mechanics["winner_allowed"] is False
assert mechanics["validity_gates"]["zero_model_calls"] is True
assert mechanics["validity_gates"]["zero_network_calls"] is True


### 8. Interpret latency without fooling ourselves

The first row often appears slower because it pays initialization and cold
context costs. That does not prove a pair is faster than a single harness.
A paid protocol should isolate setup, randomize or counterbalance blocks,
record cache state, and compare task-paired effects. The mechanics artifact
exposes this confound instead of smoothing it away.


In [ ]:
for row in mechanics["rows"]:
    print(
        row["strategy"],
        {"calls": row["calls"], "latency_ms": row["wall_latency_ms"],
         "checks": f"{row['control_checks_passed']}/{row['control_checks_total']}"},
    )
print(mechanics["claim_boundary"])


## Part IV — optionally run two matched frontier-model smoke arms

### 9. Prepare proposals without making paid calls

Set `ADVANCED_EVALUATION_LIVE=1` before starting the kernel to enable this
path. Secret entry and spend approval are separate events. Preparing the
pair writes two exact network proposals and must report zero paid calls.

```mermaid
sequenceDiagram
  participant H as Human evaluator
  participant M as MemoRizz
  participant G as GPT-5.5
  participant O as Claude Opus 5
  H->>M: prepare matched tasks
  M-->>H: two pending proposal IDs; zero calls
  H->>H: inspect hashes, context, budgets, verification
  H->>M: approve both exact envelopes
  M->>G: one Responses API call
  M->>O: one Messages API call
  M-->>H: two verified rows + parity checks
  Note over H: integration smoke only; winner disallowed
```


In [ ]:
if EVALUATION_LIVE:
    smoke_pending = evaluation_course.start_model_panel()
    pprint({
        "paid_calls_made": smoke_pending["paid_calls_made"],
        "protocol_fingerprint": smoke_pending["protocol_fingerprint"],
        "proposal_ids": smoke_pending["proposal_ids"],
    })
    assert smoke_pending["paid_calls_made"] == 0
else:
    smoke_pending = None
    print("SKIPPED: paid smoke arms require ADVANCED_EVALUATION_LIVE=1 and hidden key entry.")


### 10. Consume both proposals once and audit parity

The typed phrase is a spend decision, not a model instruction. The result
checks identical protocol, scoped context, workspace, and host verification.
A failed arm remains failed; the notebook does not substitute a cheaper or
more available model and quietly change the experiment.


In [ ]:
if EVALUATION_LIVE:
    approval_phrase = input("Type RUN FAIR PAIRED SMOKE to approve two paid calls: ").strip()
    if approval_phrase != "RUN FAIR PAIRED SMOKE":
        raise RuntimeError("The paid smoke comparison was not approved")
    smoke = evaluation_course.approve_model_panel(
        smoke_pending["proposal_ids"],
        approver_id="notebook-evaluation-owner",
        reason="Matched two-arm smoke protocol and spend reviewed.",
    )
    pprint(smoke["evaluation"])
    assert all(smoke["evaluation"]["fairness_checks"].values())
    assert smoke["evaluation"]["winner_allowed"] is False
else:
    smoke = None
    print("SKIPPED: no provider calls were made in the saved course output.")


## Part V — connect the smoke test to the full factorial

### 11. What the two-arm smoke can and cannot establish

The smoke proves current credentials, exact requested model IDs, adapter
normalization, memory grounding, approval, and host verification. It does
not estimate wrapper overhead because it has no direct arms. It does not
estimate coordination because it has no mandatory panel arm. It does not
estimate routing because it has one task, and it does not estimate provider
sensitivity because it uses one memory provider.

The full 12-arm protocol closes those design gaps, but a credible broader
claim still needs a larger preregistered task distribution, dated model
snapshots, repeated blocks across time, complete failed-call accounting,
and confidence intervals over task-level paired effects.

```mermaid
flowchart BT
  Wiring[Provider wiring works] --> Contract[Governed envelope works]
  Contract --> Fixture[One fixture result]
  Fixture --> Suite[Held-out independent task suite]
  Suite --> Domain[Bounded domain claim]
  Domain --> General[General superiority claim]
  Smoke[This notebook's paid option] --> Wiring
  Smoke --> Contract
  Smoke -. insufficient .-> Domain
  Mechanics[Deterministic audit] --> Contract
  Mechanics -. no model quality signal .-> Fixture
```


In [ ]:
lessons = source_notebook_lessons()
pprint(lessons)
assert lessons["reference"].endswith("06_fair_harness_comparison_results.ipynb")
assert "installed PyPI wheel" in lessons["runtime_import_policy"]


## Part VI — validity threats, references, and cleanup

### 12. Validity checklist

| Validity dimension | Remaining threat | Required mitigation |
|---|---|---|
| Internal | Time drift, warm caches, retries | Counterbalanced blocks, cache telemetry, retain all failures |
| Construct | Judge rewards style or provider identity | Deterministic task checks, blind judge, human disagreement review |
| External | Four synthetic fintech tasks | Several held-out task families sampled from deployment traffic |
| Statistical conclusion | Correlated reruns | Task-level paired effects and uncertainty intervals |
| Reproducibility | Floating models and environments | Dated snapshots, CLI/SDK versions, code/data hashes, lock file |
| Cost | Failed calls missing from totals | Durable pre/post-call accounting and explicit unknown-cost lower bounds |

### References and further reading

- The supplied MemoRizz example,
  `examples/metaharness/06_fair_harness_comparison_results.ipynb`, is the
  methodological source for the protocol controls adapted here.
- Jimenez et al., [*SWE-bench: Can Language Models Resolve Real-World GitHub Issues?*](https://arxiv.org/abs/2310.06770),
  2023, motivates executable repository-level acceptance tests.
- Liu et al., [*AgentBench: Evaluating LLMs as Agents*](https://arxiv.org/abs/2308.03688),
  2023, illustrates evaluation across distinct interactive environments.
- Zheng et al., [*Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*](https://arxiv.org/abs/2306.05685),
  2023, documents judge-based evaluation and known position/style effects.
- Demšar, [*Statistical Comparisons of Classifiers over Multiple Data Sets*](https://jmlr.org/papers/v7/demsar06a.html),
  JMLR 2006, explains why conclusions should be made across independent
  datasets/tasks rather than pseudo-replicated reruns.

The references motivate the design. The artifact's hashes, task-native
checks, accounting, and explicit claim boundary are the evidence for this
particular execution.


In [ ]:
import shutil

evaluation_course.close()
shutil.rmtree(EVAL_ROOT, ignore_errors=True)
print("Removed disposable evaluation resources.")


## Final takeaways

- Freeze the protocol before paid results exist.
- Compare matched arms that isolate one causal question.
- Make executable task checks primary and LLM judges diagnostic.
- Count failed calls, retries, tokens, latency, and unknown-cost gaps.
- Use independent tasks as the sample unit.
- Keep mechanics evidence, integration smoke evidence, and model-quality evidence separate.
- Preserve `winner_allowed=false` until the task distribution and analysis justify a bounded winner claim.
